## Data Processing – Replicating “I Will Survive: Predicting Business Failures from Customer Ratings”

The following pipeline mirrors the data preparation and analysis steps of the Marketing Science case study and stores them in a Pickle file for further analysis.


In [ ]:
import sys
from pathlib import Path

# add projects root directory to the system path to enable importing custom modules (e.g., from the "helpers" folder).
sys.path.append(str(Path("..").resolve()))

# Imports
import pandas as pd
import numpy as np


SEED = 42  # random seed for reproducability

np.random.seed(SEED)

In [22]:
from constants import DATA_FOLDER

# Dataframes
reviews = pd.read_csv(DATA_FOLDER / "reviews.csv")
business_covariates = pd.read_csv(DATA_FOLDER / "business_covariates.csv")

In [23]:
# create indices for training evaluation and calibration
indices = np.random.permutation(len(business_covariates))
indices_val_cal = np.random.permutation(
    np.arange(500, len(business_covariates))
)  # range 500-921 (because of sorting)


train_indices = indices[:500]  # take 500 random samples
calibration_indices = indices_val_cal[:100]  # take 100 random out of range 500-921
eval_indices = indices_val_cal[100:]  # take 321 random out range 500-921

eval_cal_indices = np.concatenate([eval_indices, calibration_indices])
eval_cal_indices_sorted = np.sort(eval_cal_indices)

In [24]:
assert (business_covariates.get("TRAIN")).sum() == 0, "training set already assigned!"

# create indices for training evaluation and calibration
indices = np.random.permutation(len(business_covariates))
indices_val_cal = np.random.permutation(
    np.arange(500, len(business_covariates))
)  # range 500-921 (because of sorting)


train_indices = indices[:500]  # take 500 random samples
calibration_indices = indices_val_cal[:100]  # take 100 random out of range 500-921
eval_indices = indices_val_cal[100:]  # take 321 random out range 500-921

# set 'TRAIN' variable to 1 for train_indices, 0 otherwise
business_covariates.loc[train_indices, "TRAIN"] = 1

# sort business_covariates so that rows with Train==1 come first
business_covariates = business_covariates.sort_values(
    by="TRAIN", ascending=False
).reset_index(
    drop=True
)  # it is possible to retreive all training data with :N_train

n_train = len(train_indices)  # number of training samples

print(f"Train/Calibration/Eval indices created:")
print(f"  Train: {len(train_indices)} samples")
print(f"  Calibration: {len(calibration_indices)} samples")
print(f"  Eval: {len(eval_indices)} samples")

business_covariates

Train/Calibration/Eval indices created:
  Train: 500 samples
  Calibration: 100 samples
  Eval: 321 samples


,business_id,name,neighborhood,address,city,state,postal_code,latitude,longitude,stars,...,chain,density,TRAIN,category,FT,Price.Level,Restaurant.Size,Number.of.Seats,ZRI,Distance.To.City.Centre
0,krf5clVkTK7ROdvp1PCsSg,"""Atami Sushi""",NaN,"""13849 N 19th Ave""",Phoenix,AZ,85023.0,33.612328,-112.099276,4.0,...,0,6,1,Asian,False,2.0,NaN,NaN,1430.0,18070.556582
1,2X07EuED0jY5C5hKQovfBA,"""Cafe At Desert Ridge""",NaN,"""20910 N Tatum Blvd, Ste 100""",Phoenix,AZ,85050.0,33.674735,-111.980063,4.0,...,0,14,1,American,False,2.0,169.0,68.0,1638.0,26340.706143
2,Ayt0hEO4siFH8h4lsCa4VQ,"""Wildflower Bread Company""",NaN,"""3800 E Sky Harbor Blvd""",Phoenix,AZ,85034.0,33.435634,-111.997900,4.0,...,1,20,1,American,False,1.0,176.0,76.0,1413.0,7268.788970
3,ViErpcikhbAjsDcpdAfwSQ,"""Maui Dog""",NaN,"""3538 E Indian School Rd""",Phoenix,AZ,85018.0,33.495447,-112.005086,4.5,...,0,9,1,American,False,1.0,132.0,62.0,1892.0,8050.797078
4,GVkTLK1rnASfW7ia7YgdKw,"""Cheba Hut - Phoenix""",NaN,"""825 N 7th St, Ste 101""",Phoenix,AZ,85006.0,33.457317,-112.064800,4.0,...,0,32,1,American,False,1.0,869.0,574.0,1369.0,1068.068259
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
916,bJeVUZ8vCVLGYGIaVbY4Xw,"""McDonald's""",NaN,"""10230 N 32nd St""",Phoenix,AZ,85028.0,33.580009,-112.013392,1.5,...,1,7,0,Fast Food,False,1.0,403.0,292.0,2007.0,15378.012512
917,ibOX3CypYVz0nJhCN5Wmcw,"""Taqueria El Vaporcito""",NaN,"""2775 W Thomas Rd""",Phoenix,AZ,85017.0,33.479810,-112.118854,3.0,...,0,4,0,Mexican,False,2.0,201.0,82.0,1223.0,5242.544637
918,e0CehK0lUWqd530L4-ni3A,"""Nuevo Burrito""",NaN,"""8202 W Indian School Rd""",Phoenix,AZ,85033.0,33.494345,-112.235229,4.0,...,0,2,0,Mexican,False,1.0,199.0,96.0,1253.0,15730.395926
919,HRAJCzcUUE3k_7lwKcyiGA,"""Vitamin T""",NaN,"""1 E Washington St, Ste 175""",Phoenix,AZ,85004.0,33.448327,-112.073579,3.0,...,0,57,0,Mexican,False,1.0,NaN,NaN,1495.0,341.659240


In [25]:
# initialize new lists

ratings = []
sentiment = []
days = []
time = []
age = []

business_ids = business_covariates["business_id"].values

# convert date string into datetime object
reviews["date"] = pd.to_datetime(reviews["date"])

for k, business_id in enumerate(business_ids):
    if k % 100 == 0:
        print(f"[{k}] - Conversion for business_id: {business_id}")

    # get temporary dataframe of all reviews with given business_id and assign column 'Number' (Rating 0, ..., M_i)
    df_temp = (
        reviews[reviews["business_id"] == business_id]
        .reset_index(drop=True)
        .assign(Number=lambda x: x.index)
    )

    # calculate days since first review
    df_temp["Days"] = (df_temp["date"] - df_temp["date"].iloc[0]).dt.days
    days.extend(df_temp["Days"].tolist())
    sentiment.extend(df_temp["sentimenttext"].tolist())
    ratings.extend(df_temp["stars"].tolist())
    time.append(len(df_temp))
    age.append(df_temp["Days"].iloc[-1])


# validation checks
assert sum(time) == len(sentiment)
assert len(days) == len(sentiment)
assert len(ratings) == len(sentiment)
print("Done ... validation checks passed!")

[0] - Conversion for business_id: krf5clVkTK7ROdvp1PCsSg
[100] - Conversion for business_id: vpDutYDhh3UTBl9W1fgM6Q
[200] - Conversion for business_id: SvoIBKec8s1kOARX_3EKXw
[300] - Conversion for business_id: QcV8KIPKbGRnWQPql55bKQ
[400] - Conversion for business_id: PSY6Fpn6QRC-yWcz0Jnm1w
[500] - Conversion for business_id: Edq3REDbBfss0WQu2Ta93w
[600] - Conversion for business_id: yzjQvTWiinB8pmbCMu8vuQ
[700] - Conversion for business_id: cbm9sq6Wis6T_WmbiFy8Ag
[800] - Conversion for business_id: FTUkGtUgd38pOx71SaVovg
[900] - Conversion for business_id: ptWNY_h088kmKhsL-gaOEg
Done ... validation checks passed!


In [26]:
assert (
    not "Age" in business_covariates
), "The key Age is already added to dataframe! Make sure, that you only run this cell once!"

# add restaurant age in days to dataframe
business_covariates["Age"] = age

# R: mutate(l_age = log(Age), Checkin = Checkin/Age*28)
# change checkin count to checkin rates (number of checkins every month, assuming a month contains 28 days)
business_covariates["Checkin"] = (
    business_covariates["Checkin"] / business_covariates["Age"] * 28
)

# add log of age to dataframe for later analysis (same as R: l_age)
business_covariates["logAge"] = np.log(business_covariates["Age"])

# R: Covariates <- covariates_business %>%
#    select(density, Checkin, category, chain, Price.Level, Restaurant.Size, Number.of.Seats, ZRI, Age)
# only get relevant covariates for analysis (must match R order and selection)
relevant_covariates = business_covariates[
    [
        "density",
        "Checkin",
        "category",
        "chain",
        "Price.Level",
        "Restaurant.Size",
        "Number.of.Seats",
        "ZRI",
        "Age",
    ]
].copy()

relevant_covariates

,density,Checkin,category,chain,Price.Level,Restaurant.Size,Number.of.Seats,ZRI,Age
0,6,0.864338,Asian,0,2.0,NaN,NaN,1430.0,1231
1,14,5.555906,American,0,2.0,169.0,68.0,1638.0,635
2,20,2.688213,American,1,1.0,176.0,76.0,1413.0,1052
3,9,9.854749,American,0,1.0,132.0,62.0,1892.0,716
4,32,11.843818,American,0,1.0,869.0,574.0,1369.0,461
...,...,...,...,...,...,...,...,...,...
916,7,0.920059,Fast Food,1,1.0,403.0,292.0,2007.0,2039
917,4,2.467996,Mexican,0,2.0,201.0,82.0,1223.0,1906
918,2,0.823529,Mexican,0,1.0,199.0,96.0,1253.0,1020
919,57,11.210428,Mexican,0,1.0,NaN,NaN,1495.0,537


In [27]:
# from sklearn.preprocessing import OneHotEncoder
# from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

assert (
    len(relevant_covariates.columns) == 9
), "One Hot Coding and Scaling  already performed on dataframe!"

# R: options(na.action="na.pass")
# R: cov_mat <- model.matrix(formula(paste("~",paste(names(Covariates),collapse = "+"),"-1")),
#                             data = Covariates)[,-8]

# The formula in R is: ~ density + Checkin + category + chain + Price.Level +
#                        Restaurant.Size + Number.of.Seats + ZRI + Age - 1
# model.matrix creates columns in this order:
# density, Checkin, categoryAmerican, categoryAsian, categoryCafes, categoryFast Food,
# categoryMexican, categoryOther, categoryPizza, categorySalad, categorySpeciality Food,
# chain, Price.Level, Restaurant.Size, Number.of.Seats, ZRI, Age
# Then [,-8] removes column 8 which is "categoryOther"

# Convert category to factor (like R)
relevant_covariates["category"] = relevant_covariates["category"].astype("category")

# Create dummy variables - R's model.matrix with "-1" creates all categories (no baseline)
relevant_covariates_encoded = pd.get_dummies(
    relevant_covariates, columns=["category"], drop_first=False, dtype=int
)

# R model.matrix order: numeric columns in original order, then categorical dummies alphabetically
# Original order: density, Checkin, category (becomes multiple), chain, Price.Level,
#                 Restaurant.Size, Number.of.Seats, ZRI, Age

# First two numeric columns before category
first_numeric = ["density", "Checkin"]

# Category dummies (alphabetically sorted)
category_cols = sorted(
    [col for col in relevant_covariates_encoded.columns if col.startswith("category_")]
)

# Remaining numeric columns after category in original order
remaining_numeric = [
    "chain",
    "Price.Level",
    "Restaurant.Size",
    "Number.of.Seats",
    "ZRI",
    "Age",
]

# Combine in R's order
column_order = first_numeric + category_cols + remaining_numeric
relevant_covariates = relevant_covariates_encoded[column_order]

# R: [,-8] removes column 8 (1-indexed in R)
# This is column index 7 in Python (0-indexed)
# Based on the order above, column 8 is "categoryOther"
if len(relevant_covariates.columns) > 7:
    col_to_remove = relevant_covariates.columns[7]
    print(f"Removing column at index 7 (R column 8): '{col_to_remove}'")

    # Verify it's categoryOther
    if "Other" in col_to_remove:
        relevant_covariates = relevant_covariates.drop(columns=[col_to_remove])
    else:
        print(f"WARNING: Expected 'categoryOther' but found '{col_to_remove}'")
        print(f"All columns: {relevant_covariates.columns.tolist()}")
        # Still remove it to match R behavior
        relevant_covariates = relevant_covariates.drop(columns=[col_to_remove])

print(f"Final covariate matrix shape: {relevant_covariates.shape}")
print(f"Columns: {relevant_covariates.columns.tolist()}")

relevant_covariates.head(1)

Removing column at index 7 (R column 8): 'category_Other'
Final covariate matrix shape: (921, 16)
Columns: ['density', 'Checkin', 'category_American', 'category_Asian', 'category_Cafes', 'category_Fast Food', 'category_Mexican', 'category_Pizza', 'category_Salad', 'category_Speciality Food', 'chain', 'Price.Level', 'Restaurant.Size', 'Number.of.Seats', 'ZRI', 'Age']


,density,Checkin,category_American,category_Asian,category_Cafes,category_Fast Food,category_Mexican,category_Pizza,category_Salad,category_Speciality Food,chain,Price.Level,Restaurant.Size,Number.of.Seats,ZRI,Age
0,6,0.864338,0,1,0,0,0,0,0,0,0,2.0,NaN,NaN,1430.0,1231


In [28]:
# R: preProc <- preProcess(cov_mat[1:500,], c("center","medianImpute"))
# Python equivalent: First fit on training data, then center, then impute
# Note: R's caret::preProcess with c("center", "medianImpute") first centers, then imputes with median

imputer = SimpleImputer(strategy="median")
scaler = StandardScaler(with_std=False)  # only centering, no scaling!


X_train = relevant_covariates.iloc[:n_train].copy()

# R applies: preProcess(cov_mat[1:500,], c("center","medianImpute"))
# This means: calculate center from training data, then impute missing values with median
# In R's caret, the order in the vector matters - "center" is applied first to calculate statistics
# but "medianImpute" fills NAs before centering is applied in the transform step

# First fit imputer on training data (to get medians for each column)
imputer.fit(X_train)

# Then fit scaler on imputed training data (to get means for centering)
X_train_imputed = imputer.transform(X_train)
scaler.fit(X_train_imputed)

# Now apply both transformations to all data
cov_mat_imputed = imputer.transform(relevant_covariates)
cov_mat_preprocessed = scaler.transform(cov_mat_imputed)

X_train_preprocessed = cov_mat_preprocessed[:n_train]

# QR-decomposition (same as R: qr() function)
Q, R = np.linalg.qr(X_train_preprocessed)

# R: Q <- qr.Q(QR)*sqrt(N_train-1)
# R: R <- qr.R(QR)/sqrt(N_train-1)
Q_scaled = Q * np.sqrt(n_train - 1)
R_scaled = R / np.sqrt(n_train - 1)

X_test = cov_mat_preprocessed[n_train:]

display(X_train)
display(pd.DataFrame(cov_mat_preprocessed))

,density,Checkin,category_American,category_Asian,category_Cafes,category_Fast Food,category_Mexican,category_Pizza,category_Salad,category_Speciality Food,chain,Price.Level,Restaurant.Size,Number.of.Seats,ZRI,Age
0,6,0.864338,0,1,0,0,0,0,0,0,0,2.0,NaN,NaN,1430.0,1231
1,14,5.555906,1,0,0,0,0,0,0,0,0,2.0,169.0,68.0,1638.0,635
2,20,2.688213,1,0,0,0,0,0,0,0,1,1.0,176.0,76.0,1413.0,1052
3,9,9.854749,1,0,0,0,0,0,0,0,0,1.0,132.0,62.0,1892.0,716
4,32,11.843818,1,0,0,0,0,0,0,0,0,1.0,869.0,574.0,1369.0,461
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,15,9.701754,1,0,0,0,0,0,0,0,0,2.0,NaN,NaN,1892.0,456
496,11,3.152174,1,0,0,0,0,0,0,0,1,1.0,378.0,138.0,1469.0,1288
497,3,2.251748,0,0,1,0,0,0,0,0,0,1.0,123.0,38.0,1253.0,572
498,11,2.222222,0,0,0,0,0,0,0,0,0,2.0,111.0,30.0,1469.0,378


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,-6.094,-4.765964,-0.244,0.882,-0.088,-0.128,-0.16,-0.098,-0.04,-0.056,-0.304,0.552,-53.25,-26.956,-63.534,-137.428
1,1.906,-0.074396,0.756,-0.118,-0.088,-0.128,-0.16,-0.098,-0.04,-0.056,-0.304,0.552,-51.75,-27.956,144.466,-733.428
2,7.906,-2.942089,0.756,-0.118,-0.088,-0.128,-0.16,-0.098,-0.04,-0.056,0.696,-0.448,-44.75,-19.956,-80.534,-316.428
3,-3.094,4.224447,0.756,-0.118,-0.088,-0.128,-0.16,-0.098,-0.04,-0.056,-0.304,-0.448,-88.75,-33.956,398.466,-652.428
4,19.906,6.213516,0.756,-0.118,-0.088,-0.128,-0.16,-0.098,-0.04,-0.056,-0.304,-0.448,648.25,478.044,-124.534,-907.428
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
916,-5.094,-4.710243,-0.244,-0.118,-0.088,0.872,-0.16,-0.098,-0.04,-0.056,0.696,-0.448,182.25,196.044,513.466,670.572
917,-8.094,-3.162306,-0.244,-0.118,-0.088,-0.128,0.84,-0.098,-0.04,-0.056,-0.304,0.552,-19.75,-13.956,-270.534,537.572
918,-10.094,-4.806772,-0.244,-0.118,-0.088,-0.128,0.84,-0.098,-0.04,-0.056,-0.304,-0.448,-21.75,0.044,-240.534,-348.428
919,44.906,5.580127,-0.244,-0.118,-0.088,-0.128,0.84,-0.098,-0.04,-0.056,-0.304,-0.448,-53.25,-26.956,1.466,-831.428


In [29]:
from helpers import comp_entropy

# aggregate review stats
# R column names: VAR, MEAN, ENTR, COUNT, ONE_STAR, TWO_STAR, THREE_STAR, FOUR_STAR, FIVE_STAR
review_stats = (
    reviews.groupby("business_id")
    .agg(
        VAR=("stars", "var"),
        MEAN=("stars", "mean"),
        ENTR=("stars", lambda x: comp_entropy(x)),
        COUNT=("stars", "size"),
        ONE_STAR=("stars", lambda x: (x == 1).sum()),
        TWO_STAR=("stars", lambda x: (x == 2).sum()),
        THREE_STAR=("stars", lambda x: (x == 3).sum()),
        FOUR_STAR=("stars", lambda x: (x == 4).sum()),
        FIVE_STAR=("stars", lambda x: (x == 5).sum()),
    )
    .reset_index()
)

# mutate count into probabilities (same as R)
for col in ["ONE_STAR", "TWO_STAR", "THREE_STAR", "FOUR_STAR", "FIVE_STAR"]:
    review_stats[col] = review_stats[col] / review_stats["COUNT"]

# Add variation coeffient to review_stats
review_stats["COV"] = np.sqrt(review_stats["VAR"]) / review_stats["MEAN"]

# R: select(business_id, density, Checkin, category, chain, Price.Level,
#           Restaurant.Size, Number.of.Seats, ZRI, Distance.To.City.Centre, Age, is_open)
benchmark_covariates = business_covariates[
    [
        "business_id",
        "density",
        "Checkin",
        "category",
        "chain",
        "Price.Level",
        "Restaurant.Size",
        "Number.of.Seats",
        "ZRI",
        "Distance.To.City.Centre",
        "Age",
        "is_open",
    ]
].copy()

# R: mutate(Closed = 1-is_open)
benchmark_covariates["Closed"] = 1 - benchmark_covariates["is_open"]

# R: left_join(temp, by="business_id")
benchmark_covariates = benchmark_covariates.merge(
    review_stats, on="business_id", how="left"
)

# R: mutate(l_COUNT = log(COUNT), category = factor(category), Closed = factor(Closed))
benchmark_covariates["l_COUNT"] = np.log(benchmark_covariates["COUNT"])

# Convert to categorical (same as R factors)
benchmark_covariates["category"] = benchmark_covariates["category"].astype("category")

# R: Closed <- fct_recode(Closed, "Closed" = "1", "Open" = "0")
# Convert Closed to categorical with proper labels
benchmark_covariates["Closed"] = (
    benchmark_covariates["Closed"].map({1: "Closed", 0: "Open"}).astype("category")
)

print(f"Benchmark covariates prepared with shape: {benchmark_covariates.shape}")

benchmark_covariates

Benchmark covariates prepared with shape: (921, 24)


,business_id,density,Checkin,category,chain,Price.Level,Restaurant.Size,Number.of.Seats,ZRI,Distance.To.City.Centre,...,MEAN,ENTR,COUNT,ONE_STAR,TWO_STAR,THREE_STAR,FOUR_STAR,FIVE_STAR,COV,l_COUNT
0,krf5clVkTK7ROdvp1PCsSg,6,0.864338,Asian,0,2.0,NaN,NaN,1430.0,18070.556582,...,3.793103,1.423186,29,0.103448,0.068966,0.137931,0.310345,0.379310,0.347898,3.367296
1,2X07EuED0jY5C5hKQovfBA,14,5.555906,American,0,2.0,169.0,68.0,1638.0,26340.706143,...,4.243902,1.135463,41,0.000000,0.073171,0.097561,0.341463,0.487805,0.215835,3.713572
2,Ayt0hEO4siFH8h4lsCa4VQ,20,2.688213,American,1,1.0,176.0,76.0,1413.0,7268.788970,...,4.235294,1.165179,68,0.044118,0.073529,0.088235,0.191176,0.602941,0.274011,4.219508
3,ViErpcikhbAjsDcpdAfwSQ,9,9.854749,American,0,1.0,132.0,62.0,1892.0,8050.797078,...,4.457447,0.992601,94,0.010638,0.031915,0.074468,0.255319,0.627660,0.190887,4.543295
4,GVkTLK1rnASfW7ia7YgdKw,32,11.843818,American,0,1.0,869.0,574.0,1369.0,1068.068259,...,4.069444,1.282947,72,0.055556,0.097222,0.083333,0.250000,0.513889,0.301177,4.276666
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
916,bJeVUZ8vCVLGYGIaVbY4Xw,7,0.920059,Fast Food,1,1.0,403.0,292.0,2007.0,15378.012512,...,1.531915,0.789908,47,0.787234,0.042553,0.085106,0.021277,0.063830,0.756026,3.850148
917,ibOX3CypYVz0nJhCN5Wmcw,4,2.467996,Mexican,0,2.0,201.0,82.0,1223.0,5242.544637,...,2.911765,1.546091,34,0.294118,0.176471,0.088235,0.205882,0.235294,0.550325,3.526361
918,e0CehK0lUWqd530L4-ni3A,2,0.823529,Mexican,0,1.0,199.0,96.0,1253.0,15730.395926,...,4.086957,1.105530,23,0.130435,0.043478,0.086957,0.086957,0.652174,0.360762,3.135494
919,HRAJCzcUUE3k_7lwKcyiGA,57,11.210428,Mexican,0,1.0,NaN,NaN,1495.0,341.659240,...,2.983051,1.544431,59,0.135593,0.220339,0.271186,0.271186,0.101695,0.405781,4.077537


In [30]:
from helpers import ModelData
from constants import PROCESSED_DATA_FOLDER

# Create ModelData instance with all data in one place
model_data = ModelData(
    n_states=None,  # not used yet, reserved for HMM models
    n_total=len(time),
    n_train=n_train,
    n_obs=int(np.sum(time)),
    n_covs=cov_mat_preprocessed.shape[1],
    time=time,
    closed=1 - business_covariates["is_open"].values,
    days=days,
    ratings=ratings,
    sentiment=sentiment,
    Q=Q_scaled,
    R=R_scaled,
    X_test=X_test,
    imputer=imputer,
    scaler=scaler,
    train_indices=np.arange(500),
    calibration_indices=calibration_indices,
    eval_indices=eval_indices,
    business_covariates=business_covariates,
    cov_mat=cov_mat_preprocessed,
    benchmark_covariates=benchmark_covariates,
)

# save to pickle
output_path = PROCESSED_DATA_FOLDER / f"processed_data_{SEED}.pkl"
model_data.to_pickle(output_path)

print(f"Data saved to {output_path}")
print(model_data.summary())

Data saved to /Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/data/processed/processed_data_123.pkl

ModelData Summary:
States (HMM): Not set
Total businesses: 921
Training samples: 500
Total observations: 64887
Number of covariates: 16

Data shapes:
- Ratings: 64887
- Sentiment: 64887
- Days: 64887
- Q matrix: (500, 16)
- R matrix: (16, 16)
- X_test: (421, 16)

Business status:
- Closed: 225
- Open: 696

Preprocessing artifacts: Available
- Train indices: 500
- Calibration indices: 100
- Eval indices: 321

Benchmark data: Available
- Benchmark covariates shape: (921, 24)
- Columns: business_id, density, Checkin, category, chain...

